<h1>Train Test Split

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import pandas as pd
df_u_clean = pd.read_csv(r'/content/drive/MyDrive/df_u_clean.csv')


In [3]:
from sklearn.model_selection import train_test_split

X = df_u_clean['narrative']
y = df_u_clean['Timely response?']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

<H1>1. Random Undersampling

In [4]:
import pandas as pd

# Combine X_train and y_train into one dataframe temporarily
train_df = pd.concat(
    [X_train.reset_index(drop=True),
     y_train.reset_index(drop=True)],
    axis=1
)

# Separate the classes
yes_df = train_df[train_df['Timely response?'] == 'Yes']
no_df = train_df[train_df['Timely response?'] == 'No']

print("Before undersampling:")
print(train_df['Timely response?'].value_counts())

# Random undersampling
yes_under = yes_df.sample(
    n=len(no_df),
    random_state=42
)

# Combine and shuffle
train_under = pd.concat([yes_under, no_df])

train_under = train_under.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nAfter undersampling:")
print(train_under['Timely response?'].value_counts())

# Separate X and y again
X_train_under = train_under['narrative']
y_train_under = train_under['Timely response?']

<H3>Encoding using sentence-transformers/all-MiniLM-L6-v2 model on X_train X_test

In [5]:
!pip install sentence_transformers

In [6]:
! pip install scikit-learn

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Initialize the embedding model
# This will automatically download the model on the first run and cache it
print("Loading MiniLM model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Convert text to embeddings
# NOTE: sentence-transformers expects a list of strings. 
# If X_train_under is a Pandas Series, convert it using .tolist()
print("Generating embeddings for the training set (this might take a moment)...")
X_train_text = X_train_under.tolist() if hasattr(X_train_under, 'tolist') else list(X_train_under)
X_train_embeddings = embedding_model.encode(X_train_text, show_progress_bar=True)

print("Generating embeddings for the test set...")
X_test_text = X_test.tolist() if hasattr(X_test, 'tolist') else list(X_test)
X_test_embeddings = embedding_model.encode(X_test_text, show_progress_bar=True)

# Quick sanity check on shapes
print(f"\nTraining embeddings shape: {X_train_embeddings.shape}") 
# Expected shape: (number_of_rows, 384)


In [8]:
from sklearn.preprocessing import LabelEncoder

# Initialize the encoder
le = LabelEncoder()

# Fit and transform your target labels
y_train_encoded = le.fit_transform(y_train_under)
y_test_encoded = le.transform(y_test)

# Now use y_train_encoded and y_test_encoded in your classifier!

<h1>3. Model training & testing

In [13]:
!pip install dagshub mlflow

In [16]:
import dagshub
import mlflow

mlflow.set_tracking_uri('')
dagshub.init(repo_owner='', repo_name='', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("undersample_model_urgent-")


In [18]:
import mlflow
import mlflow.sklearn
import logging
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# LOGGING CONFIG
# ==========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==========================================
# CHANGE THESE TWO LINES ONLY
# ==========================================

from sklearn.linear_model import LogisticRegression

MODEL = LogisticRegression(
    max_iter=1000,
    random_state=42
)

MODEL_NAME = "LogisticRegression"


# ==========================================
# TRAINING
# ==========================================

logging.info("Starting MLflow Run")

with mlflow.start_run(run_name=MODEL_NAME):

    start_time = time.time()

    try:

        # ==============================
        # PARAMETERS
        # ==============================

        mlflow.log_param(
            "embedding_model",
            "all-MiniLM-L6-v2"
        )

        mlflow.log_param(
            "embedding_dimension",
            384
        )

        mlflow.log_param(
            "test_size",
            0.2
        )

        mlflow.log_param(
            "model",
            MODEL_NAME
        )

        # ==============================
        # TRAIN MODEL
        # ==============================

        logging.info(f"Training {MODEL_NAME}")

        MODEL.fit(
        X_train_embeddings,
        y_train_encoded
        )

        logging.info(
            "Training Complete"
        )

        # ==============================
        # PREDICTIONS
        # ==============================

        logging.info(
            "Generating Predictions"
        )

        y_pred = MODEL.predict(
        X_test_embeddings
        )

        # ==============================
        # METRICS
        # ==============================
        
        accuracy = accuracy_score(
              y_test_encoded,
              y_pred
              )

        precision = precision_score(
              y_test_encoded,
              y_pred
            )

        recall = recall_score(
              y_test_encoded,
              y_pred
              )

        f1 = f1_score(
            y_test_encoded,
            y_pred
            )

        # ==============================
        # LOG METRICS
        # ==============================

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # ==============================
        # SAVE MODEL
        # ==============================

        mlflow.sklearn.log_model(
            MODEL,
            "model"
        )

        end_time = time.time()

        mlflow.log_metric(
            "training_time_seconds",
            end_time - start_time
        )

        logging.info(
            f"Accuracy : {accuracy:.4f}"
        )

        logging.info(
            f"Precision : {precision:.4f}"
        )

        logging.info(
            f"Recall : {recall:.4f}"
        )

        logging.info(
            f"F1 Score : {f1:.4f}"
        )

        logging.info(
            f"Execution Time : {end_time-start_time:.2f}"
        )

    except Exception as e:

        logging.error(
            f"Error occurred : {e}",
            exc_info=True
        )

In [20]:
import mlflow
import mlflow.sklearn
import logging
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# LOGGING CONFIG
# ==========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==========================================
# CHANGE THESE TWO LINES ONLY
# ==========================================

from sklearn.svm import LinearSVC

MODEL = LinearSVC(
    random_state=42
)

MODEL_NAME = "LinearSVC"


# ==========================================
# TRAINING
# ==========================================

logging.info("Starting MLflow Run")

with mlflow.start_run(run_name=MODEL_NAME):

    start_time = time.time()

    try:

        # ==============================
        # PARAMETERS
        # ==============================

        mlflow.log_param(
            "embedding_model",
            "all-MiniLM-L6-v2"
        )

        mlflow.log_param(
            "embedding_dimension",
            384
        )

        mlflow.log_param(
            "test_size",
            0.2
        )

        mlflow.log_param(
            "model",
            MODEL_NAME
        )

        # ==============================
        # TRAIN MODEL
        # ==============================

        logging.info(f"Training {MODEL_NAME}")

        MODEL.fit(
        X_train_embeddings,
        y_train_encoded
        )

        logging.info(
            "Training Complete"
        )

        # ==============================
        # PREDICTIONS
        # ==============================

        logging.info(
            "Generating Predictions"
        )

        y_pred = MODEL.predict(
        X_test_embeddings
        )

        # ==============================
        # METRICS
        # ==============================
        
        accuracy = accuracy_score(
              y_test_encoded,
              y_pred
              )

        precision = precision_score(
              y_test_encoded,
              y_pred
            )

        recall = recall_score(
              y_test_encoded,
              y_pred
              )

        f1 = f1_score(
            y_test_encoded,
            y_pred
            )

        # ==============================
        # LOG METRICS
        # ==============================

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # ==============================
        # SAVE MODEL
        # ==============================

        mlflow.sklearn.log_model(
            MODEL,
            "model"
        )

        end_time = time.time()

        mlflow.log_metric(
            "training_time_seconds",
            end_time - start_time
        )

        logging.info(
            f"Accuracy : {accuracy:.4f}"
        )

        logging.info(
            f"Precision : {precision:.4f}"
        )

        logging.info(
            f"Recall : {recall:.4f}"
        )

        logging.info(
            f"F1 Score : {f1:.4f}"
        )

        logging.info(
            f"Execution Time : {end_time-start_time:.2f}"
        )

    except Exception as e:

        logging.error(
            f"Error occurred : {e}",
            exc_info=True
        )

In [21]:
!pip install xgboost

In [22]:
import mlflow
import mlflow.sklearn
import logging
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# LOGGING CONFIG
# ==========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==========================================
# CHANGE THESE TWO LINES ONLY
# ==========================================

from xgboost import XGBClassifier

MODEL = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

MODEL_NAME = "XGBoost"


# ==========================================
# TRAINING
# ==========================================

logging.info("Starting MLflow Run")

with mlflow.start_run(run_name=MODEL_NAME):

    start_time = time.time()

    try:

        # ==============================
        # PARAMETERS
        # ==============================

        mlflow.log_param(
            "embedding_model",
            "all-MiniLM-L6-v2"
        )

        mlflow.log_param(
            "embedding_dimension",
            384
        )

        mlflow.log_param(
            "test_size",
            0.2
        )

        mlflow.log_param(
            "model",
            MODEL_NAME
        )

        # ==============================
        # TRAIN MODEL
        # ==============================

        logging.info(f"Training {MODEL_NAME}")

        MODEL.fit(
        X_train_embeddings,
        y_train_encoded
        )

        logging.info(
            "Training Complete"
        )

        # ==============================
        # PREDICTIONS
        # ==============================

        logging.info(
            "Generating Predictions"
        )

        y_pred = MODEL.predict(
        X_test_embeddings
        )

        # ==============================
        # METRICS
        # ==============================
        
        accuracy = accuracy_score(
              y_test_encoded,
              y_pred
              )

        precision = precision_score(
              y_test_encoded,
              y_pred
            )

        recall = recall_score(
              y_test_encoded,
              y_pred
              )

        f1 = f1_score(
            y_test_encoded,
            y_pred
            )

        # ==============================
        # LOG METRICS
        # ==============================

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # ==============================
        # SAVE MODEL
        # ==============================

        mlflow.sklearn.log_model(
            MODEL,
            "model"
        )

        end_time = time.time()

        mlflow.log_metric(
            "training_time_seconds",
            end_time - start_time
        )

        logging.info(
            f"Accuracy : {accuracy:.4f}"
        )

        logging.info(
            f"Precision : {precision:.4f}"
        )

        logging.info(
            f"Recall : {recall:.4f}"
        )

        logging.info(
            f"F1 Score : {f1:.4f}"
        )

        logging.info(
            f"Execution Time : {end_time-start_time:.2f}"
        )

    except Exception as e:

        logging.error(
            f"Error occurred : {e}",
            exc_info=True
        )

In [23]:
!pip install lightgbm

In [24]:
import mlflow
import mlflow.sklearn
import logging
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# LOGGING CONFIG
# ==========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==========================================
# CHANGE THESE TWO LINES ONLY
# ==========================================

from lightgbm import LGBMClassifier

MODEL = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

MODEL_NAME = "LightGBM"



# ==========================================
# TRAINING
# ==========================================

logging.info("Starting MLflow Run")

with mlflow.start_run(run_name=MODEL_NAME):

    start_time = time.time()

    try:

        # ==============================
        # PARAMETERS
        # ==============================

        mlflow.log_param(
            "embedding_model",
            "all-MiniLM-L6-v2"
        )

        mlflow.log_param(
            "embedding_dimension",
            384
        )

        mlflow.log_param(
            "test_size",
            0.2
        )

        mlflow.log_param(
            "model",
            MODEL_NAME
        )

        # ==============================
        # TRAIN MODEL
        # ==============================

        logging.info(f"Training {MODEL_NAME}")

        MODEL.fit(
        X_train_embeddings,
        y_train_encoded
        )

        logging.info(
            "Training Complete"
        )

        # ==============================
        # PREDICTIONS
        # ==============================

        logging.info(
            "Generating Predictions"
        )

        y_pred = MODEL.predict(
        X_test_embeddings
        )

        # ==============================
        # METRICS
        # ==============================
        
        accuracy = accuracy_score(
              y_test_encoded,
              y_pred
              )

        precision = precision_score(
              y_test_encoded,
              y_pred
            )

        recall = recall_score(
              y_test_encoded,
              y_pred
              )

        f1 = f1_score(
            y_test_encoded,
            y_pred
            )

        # ==============================
        # LOG METRICS
        # ==============================

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # ==============================
        # SAVE MODEL
        # ==============================

        mlflow.sklearn.log_model(
            MODEL,
            "model"
        )

        end_time = time.time()

        mlflow.log_metric(
            "training_time_seconds",
            end_time - start_time
        )

        logging.info(
            f"Accuracy : {accuracy:.4f}"
        )

        logging.info(
            f"Precision : {precision:.4f}"
        )

        logging.info(
            f"Recall : {recall:.4f}"
        )

        logging.info(
            f"F1 Score : {f1:.4f}"
        )

        logging.info(
            f"Execution Time : {end_time-start_time:.2f}"
        )

    except Exception as e:

        logging.error(
            f"Error occurred : {e}",
            exc_info=True
        )

In [25]:
import mlflow
import mlflow.sklearn
import logging
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# LOGGING CONFIG
# ==========================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==========================================
# CHANGE THESE TWO LINES ONLY
# ==========================================

from sklearn.neural_network import MLPClassifier

MODEL = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=128,
    learning_rate="adaptive",
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
    verbose=True
)


MODEL_NAME = "MLPClassifier_2"




# ==========================================
# TRAINING
# ==========================================

logging.info("Starting MLflow Run")

with mlflow.start_run(run_name=MODEL_NAME):

    start_time = time.time()

    try:

        # ==============================
        # PARAMETERS
        # ==============================

        mlflow.log_param(
            "embedding_model",
            "all-MiniLM-L6-v2"
        )

        mlflow.log_param(
            "embedding_dimension",
            384
        )

        mlflow.log_param(
            "test_size",
            0.2
        )

        mlflow.log_param(
            "model",
            MODEL_NAME
        )

        # ==============================
        # TRAIN MODEL
        # ==============================

        logging.info(f"Training {MODEL_NAME}")

        MODEL.fit(
        X_train_embeddings,
        y_train_encoded
        )

        logging.info(
            "Training Complete"
        )

        # ==============================
        # PREDICTIONS
        # ==============================

        logging.info(
            "Generating Predictions"
        )

        y_pred = MODEL.predict(
        X_test_embeddings
        )

        # ==============================
        # METRICS
        # ==============================
        
        accuracy = accuracy_score(
              y_test_encoded,
              y_pred
              )

        precision = precision_score(
              y_test_encoded,
              y_pred
            )

        recall = recall_score(
              y_test_encoded,
              y_pred
              )

        f1 = f1_score(
            y_test_encoded,
            y_pred
            )

        # ==============================
        # LOG METRICS
        # ==============================

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # ==============================
        # SAVE MODEL
        # ==============================

        mlflow.sklearn.log_model(
            MODEL,
            "model"
        )

        end_time = time.time()

        mlflow.log_metric(
            "training_time_seconds",
            end_time - start_time
        )

        logging.info(
            f"Accuracy : {accuracy:.4f}"
        )

        logging.info(
            f"Precision : {precision:.4f}"
        )

        logging.info(
            f"Recall : {recall:.4f}"
        )

        logging.info(
            f"F1 Score : {f1:.4f}"
        )

        logging.info(
            f"Execution Time : {end_time-start_time:.2f}"
        )

    except Exception as e:

        logging.error(
            f"Error occurred : {e}",
            exc_info=True
        )